In [2]:
import pandas as pd

# Load the new dataset
new_df = pd.read_csv('Dataset_Tweets_Facts_01-Sheet1.csv')

# Inspect the first few rows and column names
print(new_df.head())
print(new_df.info())

   ID                                              Tweet        Label  \
0   1  @POTUS Biden Blunders - 6 Month Update Inflati...  Mostly-True   
1   2  @S0SickRick @Stairmaster_ @6d6f636869 Not as m...    Half-True   
2   3  THE SUPREME COURT is siding with super rich pr...    Half-True   
3   4  @POTUS Biden Blunders Broken campaign promises...  Mostly-True   
4   5  @OhComfy I agree. The confluence of events rig...  Barely-True   

                                              Reason  \
0  This tweet lists various policy concerns that ...   
1  The tweet exaggerates the historical context b...   
2  The Supreme Court did block the eviction morat...   
3  Many of these claims have been reported and cr...   
4  The claim about Americans being executed is sp...   

                                            Evidence  
0  Inflation, COVID mismanagement, and Afghanista...  
1  There were concerns about the eviction morator...  
2  The eviction moratorium was ruled unconstituti...  
3  T

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import OneClassSVM
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.linear_model import LogisticRegression
import joblib

print("--- Step 1: Training the Classification Model ---")

# Load the dataset
df = pd.read_csv('Dataset_Tweets_Facts_01-Sheet1.csv')

# Use the 'Tweet' column as the input for the classifier
X = df['Tweet']
y = df['Label']

# Load a pre-trained Sentence Transformer model
print("Loading Sentence Transformer model...")
classification_encoder = SentenceTransformer('all-mpnet-base-v2')

# Create embeddings for the tweets
print("Creating tweet embeddings...")
X_embeddings = classification_encoder.encode(X.tolist(), show_progress_bar=True)

# Train a Logistic Regression classifier
print("Training the classifier...")
classifier = LogisticRegression(max_iter=1000, class_weight='balanced')
classifier.fit(X_embeddings, y)

# Save the trained classifier and the encoder model
joblib.dump(classifier, 'tweet_classifier.joblib')
classification_encoder.save('tweet_encoder_model')

print("✅ Classification model and encoder saved successfully.")

--- Step 1: Training the Classification Model ---
Loading Sentence Transformer model...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating tweet embeddings...


Batches:   0%|          | 0/58 [00:00<?, ?it/s]

Training the classifier...
✅ Classification model and encoder saved successfully.


In [5]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 21.7 MB/s eta 0:00:00


In [6]:
import faiss
import numpy as np

print("\n--- Step 2: Building the Evidence Retrieval System ---")

# Load the encoder model we saved earlier
retrieval_encoder = SentenceTransformer('./tweet_encoder_model')

# Get unique evidence texts to avoid duplicates in our database
evidence_list = df['Evidence'].unique().tolist()

# Create embeddings for all unique evidence texts
print("Creating evidence embeddings...")
evidence_embeddings = retrieval_encoder.encode(evidence_list, show_progress_bar=True)

# Create a FAISS index
embedding_dimension = evidence_embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dimension)
index.add(np.array(evidence_embeddings).astype('float32'))

# Save the FAISS index and the list of evidence texts
faiss.write_index(index, 'evidence_index.faiss')
pd.Series(evidence_list).to_pickle('evidence_list.pkl')

print(f"✅ Evidence index with {len(evidence_list)} entries created and saved successfully.")


--- Step 2: Building the Evidence Retrieval System ---
Creating evidence embeddings...


Batches:   0%|          | 0/57 [00:00<?, ?it/s]

✅ Evidence index with 1805 entries created and saved successfully.


In [7]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

print("\n--- Step 3: Loading the Reason Generation Model ---")

# Load a pre-trained model and tokenizer (T5 is good for text-to-text tasks)
model_name = 't5-small'
tokenizer = T5Tokenizer.from_pretrained(model_name)
generative_model = T5ForConditionalGeneration.from_pretrained(model_name)

# Save them locally for the final pipeline
tokenizer.save_pretrained('./t5_tokenizer')
generative_model.save_pretrained('./t5_model')

print(f"✅ Generative model ({model_name}) and tokenizer loaded and saved successfully.")


--- Step 3: Loading the Reason Generation Model ---


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Generative model (t5-small) and tokenizer loaded and saved successfully.


In [12]:
import joblib
import faiss
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import T5ForConditionalGeneration, T5Tokenizer

# --- Load all the pre-trained components ---
print("\n--- Final Step: Loading all components for the pipeline ---")
classifier_model = joblib.load('tweet_classifier.joblib')
encoder_model = SentenceTransformer('./tweet_encoder_model')
evidence_index = faiss.read_index('evidence_index.faiss')
evidence_list = pd.read_pickle('evidence_list.pkl').tolist()
reason_tokenizer = T5Tokenizer.from_pretrained('./t5_tokenizer')
reason_model = T5ForConditionalGeneration.from_pretrained('./t5_model')
print("✅ All components loaded.")

# --- The Main Pipeline Function ---
def fact_check_tweet(tweet_text):
    """
    Takes a tweet and returns a complete fact-check analysis including
    a predicted label, ranked evidence, and a generated reason.
    """
    print("\n-------------------------------------------")
    print(f"🔎 Analyzing tweet: \"{tweet_text}\"")

    # 1. Classification: Predict the label
    tweet_embedding = encoder_model.encode([tweet_text])
    predicted_label = classifier_model.predict(tweet_embedding)[0]

    # 2. Evidence Retrieval: Find the most relevant evidence
    # Search the FAISS index for the top 1 most similar evidence item
    _, I = evidence_index.search(np.array(tweet_embedding).astype('float32'), k=1)
    retrieved_evidence = evidence_list[I[0][0]]

    # 3. Reason Generation: Create a reason based on context
    # Construct a detailed prompt for the generative model
    prompt = f"""
    Based on the following information:
    - Tweet: "{tweet_text}"
    - Predicted Label: "{predicted_label}"
    - Relevant Evidence: "{retrieved_evidence}"

    Write a brief reason explaining why the tweet was given this label.
    Reason:
    """

    # Tokenize and generate the reason
    input_ids = reason_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).input_ids
    outputs = reason_model.generate(input_ids, max_length=60, num_beams=4, early_stopping=True)
    generated_reason = reason_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # --- Display the Results ---
    print("\n--- Fact-Check Analysis Complete ---")
    print(f"Predicted Label: {predicted_label}")
    print(f"Ranked Evidence: {retrieved_evidence}")
    print(f"Generated Reason: {generated_reason}")
    print("-------------------------------------------")


# --- Let's test your vision with a new tweet! ---
new_tweet = "@JasonKhamoro @2LarryJohnson7 Biden is a racist! Kamala is too! She is Jamaican, not African! Her Great grandfather was a slave owner in Jamaica! She locked up more blacks, and kept them locked up past their sentences! She is not a friend of minorities!"
fact_check_tweet(new_tweet)

new_tweet_2 = "@MichaelPaulhei2 @13thethe Joe Bidens great-grandfather Joseph J. Biden (1828-1880) was a slave-owner and fought for the Confederate States of America"
fact_check_tweet(new_tweet_2)


--- Final Step: Loading all components for the pipeline ---
✅ All components loaded.

-------------------------------------------
🔎 Analyzing tweet: "@JasonKhamoro @2LarryJohnson7 Biden is a racist! Kamala is too! She is Jamaican, not African! Her Great grandfather was a slave owner in Jamaica! She locked up more blacks, and kept them locked up past their sentences! She is not a friend of minorities!"

--- Fact-Check Analysis Complete ---
Predicted Label: Pants-Fire
Ranked Evidence: Kamala Harris criticized Biden’s past positions during the debate but later clarified that her comments were in the context of the debate, not personal animosity.
Generated Reason: : - Tweet: "@JasonKhamoro @2LarryJohnson7 Biden is a racist! Kamala is too! She is Jamaican, not African! Kamala is too! She is Jamaican, not African! Kam
-------------------------------------------

-------------------------------------------
🔎 Analyzing tweet: "@MichaelPaulhei2 @13thethe Joe Bidens great-grandfather Joseph J. 